## ArcGIS API for Python: TEMPO Multidimensional Image Service Publishing

This notebook demonstrates how to transform data in a **RasterCollection** and publish **TEMPO** **by reference** using the **ArcGIS API for Python** (no local `arcpy` and no geoprocessing toolbox imports required).

### Step 1: Connect and Authenticate to ArcGIS Enterprise Portal

In [ ]:
import re
from os import getenv
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path

from arcgis.gis import GIS
from arcgis.raster import Raster, RasterCollection
from arcgis.raster.analytics import get_datastores, list_datastore_content, build_multidimensional_transpose, manage_multidimensional_raster, transfer_files
from arcgis.raster.functions import multidimensional_filter, raster_calculator

load_dotenv()

portal_url = getenv("PORTAL_URL")
client_id = getenv("CLIENT_ID")

gis = GIS(url=portal_url, client_id=client_id)
print(f"Hello, {gis.users.me.username}")

### Step 2: Locate and Verify existing S3 Cloud Datastore

We can use the Raster Analytics `get_datastores` helper function to query registered data stores.

In [ ]:
cloudstore_name = getenv("CLOUDSTORE_NAME")

datastores = get_datastores(gis)
cloud_datastores = datastores.search(types="cloudStore")

matching_cloudstore = [ds for ds in cloud_datastores if cloudstore_name == ds.datapath][0]
if not matching_cloudstore:
    raise ValueError(f"No cloud datastore named '{cloudstore_name}' found in datastores.")

print(f"Found cloud datastore: {matching_cloudstore}")

In [ ]:
datastore_content = list_datastore_content(matching_cloudstore)
tempo_files = [path for path in datastore_content[matching_cloudstore.path] if "TEMPO" in path]
tempo_files

### Step 3: Establish a RasterCollection

In [ ]:
# For RasterCollections, we need the filenames & timestamps

# Method for extracting metadata from filename (possibly not recommended?)
filenames = [Path(f).stem for f in tempo_files]

timestamps = [
    datetime.strptime(match.group(1), '%Y%m%dT%H%M%S').strftime('%Y-%m-%dT%H:%M:%S')
    for f in filenames
    if (match := re.search(r'(\d{8}T\d{6})Z', f))
]

print(filenames)
print(timestamps)

In [ ]:
# # Method for extracting metadata from an ArcGIS API for Python Raster Object
# filenames = [Raster(f).catalog_path.split('/')[-1][:-3] for f in tempo_files]

# timestamps = [
#     datetime.fromisoformat(Raster(f).slices[0]['StdTime'][0]).strftime('%Y-%m-%dT%H:%M:%S') 
#     for f in tempo_files
# ]

# print(filenames)
# print(timestamps)

In [ ]:
tempo_rc = RasterCollection(
  rasters=tempo_files,
  attribute_dict={
    "TEMPO_NO2": filenames,
    "StdTime": timestamps
  }
)

tempo_rc

### Step 4: Apply Raster Functions for Quality Masking

In [ ]:
# RasterCollection provides map/reduce functions, which is great for TEMPO's QA needs
def subset_and_clean(item):
    raster = item["Raster"]
    # Pass dimensionless = True to reduce overhead... we're only interested in the 2D arrays at this point anyway, but also RFTs don't like analysis with multidim rasters that have different variables
    # Alternatively, we can keep the dimensions and set `arcgis.env.match_variables` to False for multivariate analysis
    quality = multidimensional_filter(raster, variables=['/product/main_data_quality_flag'], dimensionless=True) 
    troposphere = multidimensional_filter(raster, variables=['/product/vertical_column_troposphere'], dimensionless=True)
    cloud = multidimensional_filter(raster, variables=['/support_data/eff_cloud_fraction'], dimensionless=True)

    calculation = raster_calculator(
        rasters=[quality, troposphere, cloud],
        input_names=["quality", "troposphere", "cloud"],
        expression="SetNull((quality != 0) & (cloud >= 0.5), troposphere)",
        astype="F32"
    )

    return {"raster": calculation, "TEMPO_NO2": item["TEMPO_NO2"], "StdTime": item["StdTime"]}

clean_rc = tempo_rc.map(func=subset_and_clean)
clean_rc

In [ ]:
initial_tempo = clean_rc.filter_by_attribute(field_name="ID",operator="LESS_THAN", field_values=5)
initial_tempo

In [ ]:
append_tempo = clean_rc.filter_by_attribute(field_name="ID",operator="NOT_LESS_THAN", field_values=5)
append_tempo

### Step 5: Reduce Raster Collection to Multidimensional Raster (CRF)

In [ ]:
# Reduce RasterCollection Object to an in-memory multidimensional Raster Object
mdim_raster = initial_tempo.to_multidimensional_raster(variable_field_name="TEMPO_NO2", dimension_field_names="StdTime")

# Save the multidimensional raster as an image service
mdim_raster.save(
    output_name="TEMPO",
    process_as_multidimensional=True,
    build_transpose=True
)

In [ ]:
# Raster Analytics uses the Raster Store to persist outputs as Hosted Image Services
# To keep it solely in ArcGIS API for Python, we can use transfer files to move the CRF into our cloud store

transfer_file = transfer_files(
    input_files=f"{matching_cloudstore.path}/Hosted_TEMPO.crf",
    output_datastore=matching_cloudstore.path,
    gis=gis
)

### Step 6: Publish Raster Dataset (One time)
From here, you would publish the CRF following the `publish_raster_dataset.ipynb`
This would only need to be done once, to initialize the CRF-based image service

### Step 7: Append Multdimimensional Raster (Ongoing)

In [ ]:
append_mdim_raster = append_tempo.to_multidimensional_raster(variable_field_name="TEMPO_NO2", dimension_field_names="StdTime")

# Save the multidimensional raster as an image service
append_mdim_raster.save(
    output_name="TEMPO_subset",
    process_as_multidimensional=True,
    build_transpose=True
)

In [ ]:
target_mdim_raster = gis.content.search("TEMPO")[0].layers[0]
input_mdim_raster = gis.content.search("TEMPO_subset")[0].layers[0]

# Append new slices
manage_mdim_op = manage_multidimensional_raster(
    target_multidimensional_raster=target_mdim_raster,
    manage_mode="APPEND_SLICES",
    input_multidimensional_rasters=[input_mdim_raster],
    gis=gis
)

# Rebuild transpose
build_mdim_transpose_op = build_multidimensional_transpose(
    input_multidimensional_raster=target_mdim_raster, 
    gis=gis
)

# Delete temp subset 
gis.content.search("TEMPO_subset")[0].delete()